# 10 — Neural Inertial Odometry Testing (Baseline vs Fixed Comparison)

**SIH PS 26168 — Intelligent Dead Reckoning**

Tests **both** NIO checkpoints on held-out Driver A (Session S1) and produces a side-by-side comparison.

| Checkpoint | Description | Expected σ |
|------------|-------------|------------|
| `checkpoints/inertial_odometry/inertial_odometry_best.pt` | **Baseline v1** | 8,508,032 (overflow) |
| `checkpoints/nio_fixed/nio_fixed_best.pt` | **Fixed v2** (BoundedLogVarHead) | 0.08 – 91 m |

All numbers go to `results/nio_fixed/test_metrics.json`.

In [ ]:
import os, sys, json, datetime
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

RESULTS_DIR = PROJECT_ROOT / 'results' / 'nio_fixed'
PLOTS_DIR   = PROJECT_ROOT / 'plots'   / 'nio_fixed'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from torch.utils.data import DataLoader
from src.datasets.inertial_odometry_dataset import InertialOdometryDataset
from src.models.inertial_odometry import NeuralInertialOdometry

# Test dataset: Driver A (S1) — held out, never used in training
test_ds = InertialOdometryDataset(split='test', window_size=100, stride=50)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, drop_last=False, num_workers=2)
print(f'Test windows: {len(test_ds):,} ({len(test_loader)} batches)  — Driver A, Session S1')

def evaluate_nio_checkpoint(ckpt_path, label):
    """Load a NIO checkpoint and evaluate on test set. Returns metrics dict."""
    p = Path(ckpt_path)
    if not p.exists():
        return {'error': f'NOT_FOUND: {ckpt_path}', 'label': label}

    ckpt = torch.load(p, map_location='cpu', weights_only=False)
    cfg  = ckpt.get('config', {})

    m = NeuralInertialOdometry(
        input_dim=6,
        tcn_channels=cfg.get('tcn_channels', [64, 128, 256]),
        kernel_size=cfg.get('tcn_kernel_size', 3),
        dropout=0.0,  # eval mode
        vel_loss_weight=cfg.get('velocity_loss_weight', 0.5),
    ).to(device)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()

    all_disp_err, all_vel_err, all_logvars = [], [], []
    all_gt_vel, all_pred_vel = [], []

    with torch.no_grad():
        for batch in test_loader:
            x   = batch['imu'].to(device)
            gd  = batch['disp'].to(device)
            gv  = batch['vel'].to(device)
            pd, pv, plv = m(x)
            disp_err = torch.norm(pd - gd, dim=1).cpu().numpy()
            vel_err  = torch.abs(pv[:, 0] - gv[:, 0]).cpu().numpy()
            all_disp_err.extend(disp_err)
            all_vel_err.extend(vel_err)
            all_logvars.append(plv.cpu())
            all_gt_vel.extend(gv[:, 0].cpu().numpy())
            all_pred_vel.extend(pv[:, 0].cpu().numpy())

    disp_arr = np.array(all_disp_err)
    vel_arr  = np.array(all_vel_err)
    lv_cat   = torch.cat(all_logvars, dim=0)
    u_stats  = m.compute_uncertainty_stats(lv_cat)

    gt_v  = np.array(all_gt_vel)
    pr_v  = np.array(all_pred_vel)
    corr  = float(np.corrcoef(gt_v, pr_v)[0, 1]) if len(gt_v) > 1 else float('nan')

    metrics = {
        'label':                  label,
        'checkpoint':             str(p.relative_to(PROJECT_ROOT)),
        'session':                'S1',
        'split':                  'held_out_test',
        'driver':                 'Driver A',
        'test_windows':           len(test_ds),
        'displacement_rmse_m':    round(float(np.sqrt(np.mean(disp_arr**2))), 3),
        'displacement_mae_m':     round(float(np.mean(disp_arr)),             3),
        'displacement_p50_m':     round(float(np.percentile(disp_arr, 50)),   3),
        'displacement_p95_m':     round(float(np.percentile(disp_arr, 95)),   3),
        'displacement_max_m':     round(float(np.max(disp_arr)),              3),
        'velocity_rmse_mps':      round(float(np.sqrt(np.mean(vel_arr**2))),  3),
        'velocity_mae_mps':       round(float(np.mean(vel_arr)),              3),
        'velocity_correlation':   round(corr, 4),
        'uncertainty_mean_sigma_m': round(u_stats['mean_sigma'],  4),
        'uncertainty_p95_sigma_m':  round(u_stats['p95_sigma'],   4),
        'uncertainty_max_sigma_m':  round(u_stats['max_sigma'],   4),
        'uncertainty_nan_count':    u_stats['nan_count'],
        'uncertainty_inf_count':    u_stats['inf_count'],
        'timestamp': datetime.datetime.utcnow().isoformat() + 'Z'
    }

    print(f'\n=== {label} ===')
    print(f'  Disp RMSE: {metrics["displacement_rmse_m"]} m  (MAE={metrics["displacement_mae_m"]}m P95={metrics["displacement_p95_m"]}m)')
    print(f'  Vel  RMSE: {metrics["velocity_rmse_mps"]} m/s  corr={metrics["velocity_correlation"]}')
    print(f'  Sigma mean={metrics["uncertainty_mean_sigma_m"]}m  P95={metrics["uncertainty_p95_sigma_m"]}m  max={metrics["uncertainty_max_sigma_m"]}m')
    print(f'  NaN={metrics["uncertainty_nan_count"]}  Inf={metrics["uncertainty_inf_count"]}')

    return metrics, (np.array(all_gt_vel), np.array(all_pred_vel), disp_arr)


# Evaluate baseline checkpoint
BASELINE_CKPT = PROJECT_ROOT / 'checkpoints' / 'inertial_odometry' / 'inertial_odometry_best.pt'
FIXED_CKPT    = PROJECT_ROOT / 'checkpoints' / 'nio_fixed' / 'nio_fixed_best.pt'

results = {}
plot_data = {}

if BASELINE_CKPT.exists():
    results['baseline'], plot_data['baseline'] = evaluate_nio_checkpoint(BASELINE_CKPT, 'NIO_v1_Baseline')
else:
    print(f'[SKIP] Baseline checkpoint not found: {BASELINE_CKPT}')
    results['baseline'] = {'error': 'NOT_FOUND', 'label': 'NIO_v1_Baseline'}

if FIXED_CKPT.exists():
    results['fixed'], plot_data['fixed'] = evaluate_nio_checkpoint(FIXED_CKPT, 'NIO_v2_FixedUncertainty')
else:
    print(f'[NOT_TRAINED_YET] Fixed checkpoint not found: {FIXED_CKPT}')
    print('  → Run notebook 09 (NIO training) on Lightning GPU first.')
    results['fixed'] = {'error': 'NOT_TRAINED_YET', 'label': 'NIO_v2_FixedUncertainty'}

In [ ]:
# Comparison table
print('\n=== BEFORE vs AFTER COMPARISON (Driver A — S1) ===')
print(f'{'Metric':<35} {'Baseline v1':>18} {'Fixed v2':>18} {'Change':>12}')
print('-' * 83)

comparison_metrics = [
    ('displacement_rmse_m',       'Displacement RMSE (m)'),
    ('displacement_mae_m',        'Displacement MAE (m)'),
    ('displacement_p95_m',        'Displacement P95 (m)'),
    ('velocity_rmse_mps',         'Velocity RMSE (m/s)'),
    ('velocity_correlation',      'Velocity Correlation r'),
    ('uncertainty_mean_sigma_m',  'σ mean (m)'),
    ('uncertainty_p95_sigma_m',   'σ P95 (m)'),
    ('uncertainty_max_sigma_m',   'σ max (m)'),
    ('uncertainty_nan_count',     'NaN count'),
]

for key, label in comparison_metrics:
    b_val = results['baseline'].get(key, 'N/A')
    f_val = results['fixed'].get(key, 'N/A')
    if isinstance(b_val, (int, float)) and isinstance(f_val, (int, float)) and b_val != 0:
        delta = f'{((f_val - b_val) / abs(b_val)) * 100:+.1f}%'
    else:
        delta = 'N/A'
    print(f'{label:<35} {str(b_val):>18} {str(f_val):>18} {delta:>12}')

# Write combined results
combined = {
    'experiment':  'nio_baseline_vs_fixed_comparison',
    'timestamp':   datetime.datetime.utcnow().isoformat() + 'Z',
    'session':     'S1',
    'driver':      'Driver A (held-out test)',
    'baseline':    results.get('baseline', {}),
    'fixed':       results.get('fixed', {}),
}
out_path = RESULTS_DIR / 'test_metrics.json'
with open(out_path, 'w') as f:
    json.dump(combined, f, indent=2)
print(f'\nResults saved: {out_path}')

In [ ]:
# Visualization: displacement error CDF + sigma comparison
if 'baseline' in plot_data and 'fixed' in plot_data:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Displacement error CDF
    for label, (_, _, disp_arr) in plot_data.items():
        sorted_e = np.sort(disp_arr)
        cdf = np.arange(1, len(sorted_e)+1) / len(sorted_e)
        color = 'steelblue' if label == 'baseline' else 'crimson'
        lname = 'Baseline v1' if label == 'baseline' else 'Fixed v2'
        axes[0].plot(sorted_e, cdf, color=color, label=lname)
    axes[0].set_xlabel('Displacement Error (m)'); axes[0].set_ylabel('CDF')
    axes[0].set_title('Displacement Error CDF — Driver A'); axes[0].legend(); axes[0].grid(True, alpha=0.4)

    # Velocity tracking scatter — fixed only
    gt_v, pr_v, _ = plot_data['fixed']
    axes[1].scatter(gt_v[:500], pr_v[:500], alpha=0.3, s=10, color='crimson')
    vmax = max(gt_v[:500].max(), pr_v[:500].max())
    axes[1].plot([0, vmax], [0, vmax], 'k--', label='Perfect')
    axes[1].set_xlabel('GT Speed (m/s)'); axes[1].set_ylabel('Pred Speed (m/s)')
    axes[1].set_title(f'Velocity: Fixed v2 (r={results["fixed"].get("velocity_correlation","N/A")})')
    axes[1].legend(); axes[1].grid(True, alpha=0.4)

    # Sigma comparison bar
    sigma_compare = {
        'Baseline σ mean': results['baseline'].get('uncertainty_mean_sigma_m', 0),
        'Fixed σ mean':    results['fixed'].get('uncertainty_mean_sigma_m', 0),
        'Fixed σ P95':     results['fixed'].get('uncertainty_p95_sigma_m', 0),
        'Fixed σ max':     results['fixed'].get('uncertainty_max_sigma_m', 0),
    }
    colors = ['tomato', 'steelblue', 'steelblue', 'steelblue']
    bars = axes[2].bar(sigma_compare.keys(), sigma_compare.values(), color=colors)
    axes[2].set_ylabel('Sigma (m)'); axes[2].set_title('Uncertainty σ Before/After')
    axes[2].set_yscale('log')
    axes[2].axhline(91.2, color='g', linestyle='--', label='Physical max=91.2m')
    plt.setp(axes[2].get_xticklabels(), rotation=20, ha='right', fontsize=9)
    axes[2].legend()

    plt.suptitle('NIO: Baseline v1 vs Fixed v2 — Driver A (S1)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig_path = PLOTS_DIR / 'nio_baseline_vs_fixed_comparison.png'
    plt.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f'Comparison plot saved: {fig_path}')
else:
    print('Skipping visualization — one or both checkpoints missing.')